# FERRUS CORPUS — corpus_main.ipynb
## Fregate 02 : Convertisseur FBX / GLB → .blend pour EXODUS

```
IN/  ferrus_P*.fbx  ou  ferrus_P*.glb
         ↓
   convert_to_blend.py (Blender headless)
         ↓
OUT/ corpus_P*.blend  +  rapport_corpus.json
```

**POUR L'EMPEROR. POUR LA FLOTTE FERRUS.**

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELLULE 0 — GIT SYNC
# Monte le Drive + clone/pull le repo + sync le codebase
# A executer EN PREMIER a chaque session Colab
# ═══════════════════════════════════════════════════════════

import os, shutil, subprocess
from google.colab import drive

# ── 0.1 Monter Google Drive ──────────────────────────────
drive.mount('/content/drive', force_remount=False)

# ── 0.2 CONFIGURER ICI ───────────────────────────────────
DRIVE_ROOT  = '/content/drive/MyDrive/FLOTTE-FERRUS'
GITHUB_REPO = 'https://github.com/kioka8877-ux/FLOTTE-FERRUS.git'
CLONE_DIR   = '/content/FLOTTE-FERRUS'

# ── 0.3 Clone ou Pull ────────────────────────────────────
if os.path.isdir(os.path.join(CLONE_DIR, '.git')):
    print('[GIT SYNC] Repo deja clone — git pull...')
    proc = subprocess.run(
        ['git', '-C', CLONE_DIR, 'pull', '--rebase'],
        capture_output=True, text=True
    )
    print(proc.stdout.strip() or 'Deja a jour.')
else:
    print('[GIT SYNC] Clone du repo...')
    proc = subprocess.run(
        ['git', 'clone', '--depth=1', GITHUB_REPO, CLONE_DIR],
        capture_output=True, text=True
    )
    print(proc.stdout.strip() or 'Clone OK.')
    if proc.returncode != 0:
        print('[GIT SYNC] ERREUR :', proc.stderr[-500:])
        raise RuntimeError('git clone echoue')

# ── 0.4 Sync le codebase sur Drive ───────────────────────
SRC_CODEBASE  = os.path.join(CLONE_DIR, '02_FERRUS_CORPUS', 'codebase')
DEST_CODEBASE = os.path.join(DRIVE_ROOT, '02_FERRUS_CORPUS', 'codebase')
os.makedirs(DEST_CODEBASE, exist_ok=True)

for fname in ['convert_to_blend.py']:
    src  = os.path.join(SRC_CODEBASE, fname)
    dest = os.path.join(DEST_CODEBASE, fname)
    shutil.copy2(src, dest)
    print(f'[GIT SYNC] Copie : {fname} → Drive/02_FERRUS_CORPUS/codebase/')

# ── 0.5 Creer les dossiers requis sur Drive ──────────────
for folder in ['IN', 'OUT']:
    os.makedirs(os.path.join(DRIVE_ROOT, '02_FERRUS_CORPUS', folder), exist_ok=True)

# ── 0.6 Bilan ────────────────────────────────────────────
print()
print('[GIT SYNC] ══════════════════════════════════════')
print('[GIT SYNC] Codebase synchronise depuis GitHub')
print(f'[GIT SYNC] DRIVE_ROOT : {DRIVE_ROOT}')
for folder in ['IN', 'OUT', 'codebase']:
    path = os.path.join(DRIVE_ROOT, '02_FERRUS_CORPUS', folder)
    print(f'  {"OK" if os.path.isdir(path) else "MANQUANT"}  {path}')
print('[GIT SYNC] Passer a la cellule SETUP ▶')
print('[GIT SYNC] ══════════════════════════════════════')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELLULE 1 — SETUP
# Detecte Blender + prepare les chemins
# ═══════════════════════════════════════════════════════════

import os, glob, subprocess, json

# ── 1.1 Chemins ───────────────────────────────────────────
CORPUS_ROOT    = os.path.join(DRIVE_ROOT, '02_FERRUS_CORPUS')
IN_DIR         = os.path.join(CORPUS_ROOT, 'IN')
OUT_DIR        = os.path.join(CORPUS_ROOT, 'OUT')
CONVERT_SCRIPT = os.path.join(CORPUS_ROOT, 'codebase', 'convert_to_blend.py')

os.makedirs(OUT_DIR, exist_ok=True)

# ── 1.2 Detecter Blender ─────────────────────────────────
BLENDER_CANDIDATES = [
    '/content/drive/MyDrive/EXODUS_AI_MODELS/blender-4.0.0-linux-x64/blender',
    '/content/drive/MyDrive/EXODUS_AI_MODELS/BLENDER/blender-4.0.0-linux-x64/blender',
    '/usr/local/blender/blender',
    '/content/blender/blender',
]
BLENDER_BIN = next((p for p in BLENDER_CANDIDATES if os.path.isfile(p)), None)

if not BLENDER_BIN:
    print('[SETUP] Blender non trouve — renseigner manuellement :')
    print('        BLENDER_BIN = "/chemin/vers/blender"')
else:
    print(f'[SETUP] Blender detecte : {BLENDER_BIN}')

print(f'[SETUP] IN     : {IN_DIR}')
print(f'[SETUP] OUT    : {OUT_DIR}')
print(f'[SETUP] Script : {"OK" if os.path.isfile(CONVERT_SCRIPT) else "MANQUANT — executer GIT SYNC"}')
print('[SETUP] OK')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELLULE 2 — CONFIG
# Liste les fichiers FBX et GLB dans IN/
# ═══════════════════════════════════════════════════════════

input_files = sorted(
    glob.glob(os.path.join(IN_DIR, '*.fbx')) +
    glob.glob(os.path.join(IN_DIR, '*.glb')) +
    glob.glob(os.path.join(IN_DIR, '*.gltf'))
)

if not input_files:
    print('[CONFIG] ERREUR — Aucun fichier .fbx / .glb dans IN/')
    print(f'[CONFIG] Deposer les fichiers dans : {IN_DIR}')
else:
    print(f'[CONFIG] {len(input_files)} fichier(s) trouves dans IN/ :')
    print()
    print(f'  {"FICHIER":<35} {"FORMAT":<8} {"TAILLE"}')
    print('  ' + '─' * 55)
    for f in input_files:
        ext  = os.path.splitext(f)[1].upper().lstrip('.')
        size = os.path.getsize(f) // 1024
        print(f'  {os.path.basename(f):<35} {ext:<8} {size} Ko')

ready = bool(input_files and os.path.isfile(CONVERT_SCRIPT) and BLENDER_BIN)
print()
print('[CONFIG] BILAN ──────────────────────────────────')
print(f'  Fichiers sources : {len(input_files)}')
print(f'  Script           : {"OK" if os.path.isfile(CONVERT_SCRIPT) else "MANQUANT"}')
print(f'  Blender          : {"OK" if BLENDER_BIN else "MANQUANT"}')
print(f'  GO               : {"OUI" if ready else "NON — corriger les points ci-dessus"}')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELLULE 3 — CONVERSION
# Boucle sur N fichiers FBX/GLB → .blend
# ═══════════════════════════════════════════════════════════

import datetime

if not ready:
    print('[CONVERSION] Conditions non remplies — executer CONFIG dabord')
    raise SystemExit

REPORT_PATH = os.path.join(OUT_DIR, 'rapport_corpus.json')

rapport = {
    'generated_at': datetime.datetime.now().isoformat(),
    'blender_bin': BLENDER_BIN,
    'total_files': len(input_files),
    'files': []
}

for src_path in input_files:
    stem     = os.path.splitext(os.path.basename(src_path))[0]
    out_name = stem.replace('ferrus_', 'corpus_') if stem.startswith('ferrus_') else f'corpus_{stem}'
    out_blend = os.path.join(OUT_DIR, f'{out_name}.blend')

    print(f'[CONVERSION] ─────────────────────────────────')
    print(f'[CONVERSION] Source : {os.path.basename(src_path)}')
    print(f'[CONVERSION] Sortie : {os.path.basename(out_blend)}')

    cmd = [
        BLENDER_BIN, '--background',
        '--python', CONVERT_SCRIPT,
        '--',
        '--input',  src_path,
        '--output', out_blend,
    ]

    proc = subprocess.run(cmd, capture_output=True, text=True, timeout=120)

    corpus_lines = [l for l in proc.stdout.splitlines() if '[CORPUS]' in l]
    for line in corpus_lines:
        print(line)

    if proc.returncode != 0:
        print(f'[CONVERSION] ERREUR Blender (code {proc.returncode})')
        for line in [l for l in proc.stderr.splitlines() if 'Error' in l or 'error' in l][-5:]:
            print(f'  {line}')
        rapport['files'].append({'source': os.path.basename(src_path), 'status': 'ERREUR'})
        continue

    size = os.path.getsize(out_blend) if os.path.isfile(out_blend) else 0
    rapport['files'].append({
        'source':       os.path.basename(src_path),
        'format':       os.path.splitext(src_path)[1].upper().lstrip('.'),
        'output_blend': os.path.basename(out_blend),
        'size_bytes':   size,
        'status':       'OK'
    })
    print(f'[CONVERSION] OK — {size // 1024} Ko')

with open(REPORT_PATH, 'w') as f:
    json.dump(rapport, f, indent=2)

ok  = sum(1 for x in rapport['files'] if x['status'] == 'OK')
err = len(rapport['files']) - ok
print()
print('[CONVERSION] ═══════════════════════════════════')
print(f'[CONVERSION] TERMINE — {ok}/{len(input_files)} fichier(s) convertis')
if err:
    print(f'[CONVERSION] ATTENTION — {err} erreur(s)')
print(f'[CONVERSION] Rapport : {REPORT_PATH}')
print('[CONVERSION] ═══════════════════════════════════')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELLULE 4 — RAPPORT
# Affiche le bilan de conversion
# ═══════════════════════════════════════════════════════════

REPORT_PATH = os.path.join(OUT_DIR, 'rapport_corpus.json')

if not os.path.isfile(REPORT_PATH):
    print('[RAPPORT] rapport_corpus.json introuvable — executer CONVERSION dabord')
else:
    with open(REPORT_PATH) as f:
        r = json.load(f)

    print(f'[RAPPORT] Genere le  : {r["generated_at"]}')
    print(f'[RAPPORT] Total      : {r["total_files"]} fichier(s)')
    print()
    print(f'  {"SOURCE":<35} {"FORMAT":<8} {"OUTPUT .blend":<30} {"TAILLE":<12} {"STATUT"}')
    print('  ' + '─' * 95)
    for item in r['files']:
        if item['status'] == 'OK':
            size_ko = item.get('size_bytes', 0) // 1024
            print(
                f'  {item["source"]:<35} '
                f'{item.get("format", "?"):<8} '
                f'{item["output_blend"]:<30} '
                f'{size_ko} Ko{"":<6} '
                f'OK'
            )
        else:
            print(f'  {item["source"]:<35} {"─":<8} {"─":<30} {"─":<12} ERREUR')
    print()
    print(f'[RAPPORT] Fichiers .blend dans : {OUT_DIR}')
    print('[RAPPORT] Prets pour EXODUS.')